In [1]:
# ============================= DATA PREPROCESSING ==================================

In [2]:
# A) Preprocessing (video → frames + wav)
import logging
logging.getLogger().setLevel(logging.ERROR)

import os, csv, subprocess, json
from pathlib import Path

def run(cmd):
    subprocess.run(cmd, check=True)

def preprocess_video_dataset(csv_path, out_root, fps=5, sr=16000):
    """
    Reads csv with columns: emotion, filepath, split
    For each video file, extracts:
      - frames at fps into out_root/frames/<id>/
      - audio wav 16k mono into out_root/audio/<id>.wav
    Writes a manifest jsonl with resolved paths.
    """
    out_root = Path(out_root)
    (out_root/"frames").mkdir(parents=True, exist_ok=True)
    (out_root/"audio").mkdir(parents=True, exist_ok=True)
    manifest_path = out_root/"manifest.jsonl"

    with open(csv_path, "r") as f, open(manifest_path, "w") as mf:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            video_path = row["filepath"]
            emo = row["label"]
            split = row["split"]
            sample_id = f"{Path(video_path).stem}_{i}"

            frames_dir = out_root/"frames"/sample_id
            audio_path = out_root/"audio"/f"{sample_id}.wav"
            frames_dir.mkdir(parents=True, exist_ok=True)

            # Extract frames
            # -vf fps=..., scale keeps original; we'll resize in dataloader
            run([
                "ffmpeg", "-y", "-i", video_path,
                "-vf", f"fps={fps}",
                str(frames_dir/"%05d.jpg")
            ])

            # Extract audio (mono, 16k)
            run([
                "ffmpeg", "-y", "-i", video_path,
                "-ac", "1", "-ar", str(sr),
                str(audio_path)
            ])

            record = {
                "id": sample_id,
                "emotion": emo,
                "split": split,
                "frames_dir": str(frames_dir),
                "audio_wav": str(audio_path),
                "src_video": video_path
            }
            mf.write(json.dumps(record) + "\n")

    print("Wrote manifest:", manifest_path)

# preprocess_video_dataset(os.path.join('datasets', 'crema-d', 'updated_labels.csv'), os.path.join('datasets', 'crema-d-prep'), fps=5)
# preprocess_video_dataset(os.path.join('datasets', 'ravdess', 'updated_labels.csv'), os.path.join('datasets', 'ravdess-prep'), fps=5)

In [3]:
# B) Datasets (AffectNet image-only, AV manifests)

import json, random, math
import torch
import torchaudio
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from datasets_av import AVManifestDataset

EMO_MAP = {}  # filled after scanning labels, or define fixed mapping

def build_label_map(csv_or_manifest_paths):
    labels = set()
    for p in csv_or_manifest_paths:
        p = str(p)
        if p.endswith(".csv"):
            import csv
            with open(p,"r") as f:
                r = csv.DictReader(f)
                for row in r: labels.add(row["label"])
        else:
            with open(p,"r") as f:
                for line in f:
                    labels.add(json.loads(line)["emotion"])
    labels = sorted(list(labels))
    return {lab:i for i,lab in enumerate(labels)}

In [4]:
# ======================================== UTILITY FUNCTIONS ==========================================

In [5]:
# C) Heterogeneity augmentations (novel evaluation + training)
import torch.nn.functional as F

import random
import torch

def corrupt_frames(frames, p=0.5):
    """
    frames: [T,3,H,W] or [B,T,3,H,W]
    Applies the SAME occlusion block to all frames (and all batch items).
    """
    if random.random() > p:
        return frames

    if frames.dim() == 4:
        T, C, H, W = frames.shape
        B = None
    elif frames.dim() == 5:
        B, T, C, H, W = frames.shape
    else:
        raise ValueError(f"frames must be 4D or 5D, got {frames.shape}")

    x0 = random.randint(0, W // 2)
    y0 = random.randint(0, H // 2)
    w  = random.randint(W // 8, W // 3)
    h  = random.randint(H // 8, H // 3)

    out = frames.clone()
    if frames.dim() == 4:
        out[:, :, y0:y0+h, x0:x0+w] = 0.0
    else:
        out[:, :, :, y0:y0+h, x0:x0+w] = 0.0

    return out


def corrupt_audio(wav, p=0.5, noise_std=0.02):
    # wav: [N]
    if random.random() > p:
        return wav
    noise = torch.randn_like(wav) * noise_std
    return torch.clamp(wav + noise, -1.0, 1.0)

def modality_dropout(frames, wav, p_drop=0.15):
    # drop one modality entirely
    r = random.random()
    if r < p_drop/2:
        frames = torch.zeros_like(frames)
    elif r < p_drop:
        wav = torch.zeros_like(wav)
    return frames, wav

def temporal_misalign(wav, max_shift=0.5, sr=16000, p=0.2):
    """
    wav: [N] or [B, N]
    randomly time-shifts waveform by up to ±max_shift seconds
    """
    if random.random() > p:
        return wav

    shift = int(random.uniform(-max_shift, max_shift) * sr)
    if shift == 0:
        return wav

    if wav.dim() == 1:
        # [N]
        if shift > 0:
            return torch.cat([torch.zeros(shift, device=wav.device), wav[:-shift]], dim=0)
        else:
            s = -shift
            return torch.cat([wav[s:], torch.zeros(s, device=wav.device)], dim=0)

    elif wav.dim() == 2:
        # [B, N]
        B, N = wav.shape
        if shift > 0:
            pad = torch.zeros(B, shift, device=wav.device, dtype=wav.dtype)
            return torch.cat([pad, wav[:, :-shift]], dim=1)
        else:
            s = -shift
            pad = torch.zeros(B, s, device=wav.device, dtype=wav.dtype)
            return torch.cat([wav[:, s:], pad], dim=1)

    else:
        raise ValueError(f"Expected wav dim 1 or 2, got {wav.dim()}")


@torch.no_grad()
def compute_classification_metrics(y_true, y_pred, num_classes):
    """
    y_true, y_pred: 1D torch tensors (CPU or GPU)
    Returns dict with:
      accuracy, macro_f1, macro_precision, macro_recall,
      per_class_precision/recall/f1, confusion_matrix
    """
    y_true = y_true.view(-1).to(torch.long)
    y_pred = y_pred.view(-1).to(torch.long)

    # Confusion matrix: rows=true, cols=pred
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long, device=y_true.device)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1

    tp = cm.diag().to(torch.float32)
    fp = cm.sum(dim=0).to(torch.float32) - tp
    fn = cm.sum(dim=1).to(torch.float32) - tp
    tn = cm.sum().to(torch.float32) - (tp + fp + fn)

    eps = 1e-8
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)

    accuracy = tp.sum() / (cm.sum().to(torch.float32) + eps)

    macro_precision = precision.mean()
    macro_recall    = recall.mean()
    macro_f1        = f1.mean()

    return {
        "accuracy": float(accuracy.item()),
        "macro_f1": float(macro_f1.item()),
        "macro_precision": float(macro_precision.item()),
        "macro_recall": float(macro_recall.item()),
        "per_class_precision": precision.detach().cpu().tolist(),
        "per_class_recall": recall.detach().cpu().tolist(),
        "per_class_f1": f1.detach().cpu().tolist(),
        "confusion_matrix": cm.detach().cpu(),  # tensor
    }


In [6]:
# =================================== BACKBONE MODELS + RAMER =====================================

In [7]:
# D) Model (pretrained encoders + reliability fusion)

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models

class VisionBackbone(nn.Module):
    def __init__(self, out_dim=256, train_backbone=False):
        super().__init__()
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])  # pool output [B,512,1,1]
        self.proj = nn.Linear(512, out_dim)
        self.train_backbone = train_backbone
        # Freeze everything first
        for p in self.backbone.parameters():
            p.requires_grad = False
        
        # Unfreeze only the last ResNet block (layer4)
        for name, p in self.backbone.named_parameters():
            if "7" in name:   # in Sequential(children[:-1]), layer4 is typically index 7
                p.requires_grad = True

        # Why "7"? In ResNet18, children() indices are usually:
        # 0 conv1,1 bn1,2 relu,3 maxpool,4 layer1,5 layer2,6 layer3,7 layer4,8 avgpool,9 fc
        # Since we use children()[:-1], layer4 becomes index 7.

    def forward(self, frames):
        # frames: [B,T,3,224,224]
        B,T,C,H,W = frames.shape
        x = frames.view(B*T, C, H, W)
        z = self.backbone(x).flatten(1)  # [B*T,512]
        z = self.proj(z)                 # [B*T,out_dim]
        z = z.view(B, T, -1).mean(dim=1) # temporal average [B,out_dim]
        return z

class AudioBackbone(nn.Module):
    def __init__(self, out_dim=256, train_backbone=False):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_mels=128)

        base = models.mobilenet_v2(pretrained=True)
        base.classifier = nn.Identity()   # drop last layer
        for p in base.parameters():
            p.requires_grad = False

        self.backbone = base
        self.proj = nn.Linear(1280, out_dim)

    def forward(self, wav):
        spec = self.mel(wav)              # [B,128,T]
        spec = torch.log(spec + 1e-6)
        x = self.backbone(spec.unsqueeze(1).repeat(1,3,1,1))
        return self.proj(x)

class RAMER(nn.Module):
    def __init__(self, n_classes, z_dim=256):
        super().__init__()
        self.vision = VisionBackbone(out_dim=z_dim, train_backbone=False)
        self.audio  = AudioBackbone(out_dim=z_dim, train_backbone=False)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # Fixed: rel input = logits_v(n_classes) + logits_a(n_classes) + conf(2) + d(4)
        rel_in = n_classes * 2 + 2 + 4
        self.rel = nn.Sequential(
            nn.Linear(rel_in, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2),   # r_v, r_a
        )

        nn.init.zeros_(self.rel[-1].weight)
        nn.init.zeros_(self.rel[-1].bias)

    @staticmethod
    def _disagreement(z_v, z_a, logits_v, logits_a):
        p_v = torch.softmax(logits_v, dim=-1)
        p_a = torch.softmax(logits_a, dim=-1)
        absdiff = torch.abs(p_v - p_a).mean(dim=-1, keepdim=True)  # [B,1]
        kl_va = (p_v * (p_v.clamp_min(1e-8).log() - p_a.clamp_min(1e-8).log())).sum(dim=-1, keepdim=True)
        kl_av = (p_a * (p_a.clamp_min(1e-8).log() - p_v.clamp_min(1e-8).log())).sum(dim=-1, keepdim=True)
        cos   = nn.functional.cosine_similarity(z_v, z_a, dim=-1).unsqueeze(-1)  # [B,1]
        d = torch.cat([absdiff, kl_va, kl_av, cos], dim=-1)  # [B,4]
        return d

    def forward(self, frames, wav):
        # z_v = self.vision(frames)
        # z_a = self.audio(wav)
        # logits_v = self.head_v(z_v)
        # logits_a = self.head_a(z_a)

        # d = self._disagreement(z_v, z_a, logits_v, logits_a)

        # # Unimodal confidence: max-softmax probability — direct quality signal
        # conf_v = torch.softmax(logits_v, dim=-1).max(dim=-1, keepdim=True).values  # [B,1]
        # conf_a = torch.softmax(logits_a, dim=-1).max(dim=-1, keepdim=True).values  # [B,1]
        
        # # Fixed: feed logits (not frozen features) + conf + disagreement
        # rel_in   = torch.cat([logits_v, logits_a, conf_v, conf_a, d], dim=-1)
        # r_logits = self.rel(rel_in)
        # r = torch.softmax(r_logits, dim=-1)   # [B,2]
        # r_v, r_a = r[:, 0:1], r[:, 1:2]
 
        # logits = r_v * logits_v + r_a * logits_a
        
        # return {
        #     "logits": logits,
        #     "logits_v": logits_v,
        #     "logits_a": logits_a,
        #     "r": r,
        #     "d": d
        # }
        z_v = self.vision(frames)
        z_a = self.audio(wav)
    
        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)
    
        d = self._disagreement(z_v, z_a, logits_v, logits_a)
    
        p_v = torch.softmax(logits_v, dim=-1)
        p_a = torch.softmax(logits_a, dim=-1)
    
        # Entropy: high when modality is confused (corrupted), low when confident
        ent_v = -(p_v * p_v.clamp_min(1e-8).log()).sum(dim=-1, keepdim=True)  # [B,1]
        ent_a = -(p_a * p_a.clamp_min(1e-8).log()).sum(dim=-1, keepdim=True)  # [B,1]
    
        rel_in = torch.cat([logits_v, logits_a, ent_v, ent_a, d], dim=-1)
        r = torch.softmax(self.rel(rel_in), dim=-1)
    
        return {
            "logits": r[:, 0:1] * logits_v + r[:, 1:2] * logits_a,
            "logits_v": logits_v, "logits_a": logits_a,
            "r": r, "d": d,
        }

In [8]:
# E) Training steps (full loop with heterogeneity regularizers)

from torch.utils.data import DataLoader
import torch.optim as optim

# ─────────────────────────────────────────────────────────────────────────────
# Fix 2: ramer_loss
#   - Added loss_push: pushes r asymmetrically when modalities disagree.
#     Previously both loss_ent and loss_agree only pushed r toward 0.5/0.5,
#     leaving zero gradient for asymmetric weights (RAMER's entire purpose).
#   - Removed loss_ent (ent_lambda=0.0): it directly fought loss_push.
#   - Reduced delta 0.1 → 0.02 (loss_agree was dominating).
#   - out_corr stub kept as-is (hinge is handled in train_ramer, unchanged).
# ─────────────────────────────────────────────────────────────────────────────
def ramer_loss(out, y, out_corr=None, margin=0.05,
               alpha=0.3, beta=0.3, gamma=0.5, delta=0.02,
               ent_lambda=0.0,   # removed: was pushing r toward 0.5/0.5
               agree_tau=0.2):
 
    logits, logits_v, logits_a = out["logits"], out["logits_v"], out["logits_a"]
    r = out["r"]   # [B,2]
 
    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha * F.cross_entropy(logits_v, y) + beta * F.cross_entropy(logits_a, y)
 
    p_v = torch.softmax(logits_v, dim=-1)
    p_a = torch.softmax(logits_a, dim=-1)
 
    # Symmetric KL: how much do modalities disagree?
    D = (p_v * (p_v.clamp_min(1e-8).log() - p_a.clamp_min(1e-8).log())).sum(dim=-1) + \
        (p_a * (p_a.clamp_min(1e-8).log() - p_v.clamp_min(1e-8).log())).sum(dim=-1)
 
    agree_mask    = (D <  agree_tau).float()
    disagree_mask = (D >= agree_tau).float()
 
    # loss_agree: when modalities agree, keep r balanced (unchanged intent, smaller weight)
    loss_agree = (agree_mask * ((r[:, 0] - 0.5)**2 + (r[:, 1] - 0.5)**2)).mean() * delta
 
    # loss_push: when modalities DISAGREE, push r toward the more confident modality.
    # This was MISSING — without it, the rel MLP had zero incentive to be asymmetric.
    conf_v = p_v.max(dim=-1).values   # [B]
    conf_a = p_a.max(dim=-1).values   # [B]
    # target_rv = 1 when vision more confident, 0 when audio more confident
    target_rv = (conf_v > conf_a).float()   # [B]
    # remove it temporary
    # loss_push = (disagree_mask * (target_rv - r[:, 0]).abs()).mean() * 0.3
    # loss_push = 0.0 
    # return loss_fuse + loss_uni + loss_agree + loss_push
    return loss_fuse + loss_uni + loss_agree


# ─────────────────────────────────────────────────────────────────────────────
# 3. train_ramer
#    Changes:
#    a) CE loss now runs on CLEAN pass (out_clean), not corrupted pass (out).
#       The corrupted pass was degrading classification head gradients.
#    b) Hinge weight increased from 0.5 to 1.5 so it competes with CE loss.
#    c) Everything else (augmentation order, modality_dropout, dataloader,
#       optimizer, scheduler, eval) is IDENTICAL to your original code.
# ─────────────────────────────────────────────────────────────────────────────
def train_ramer(model, dl_train, dl_val, device="cuda",
                epochs=15, lr=1e-4,
                save_path="ramer_best.pt"):
 
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
 
    best_f1 = -1.0
    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
 
        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)
            
            # Augmentation
            frames2 = corrupt_frames(frames, p=0.1)
            wav2    = corrupt_audio(wav, p=0.1)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.1)
            wav2    = temporal_misalign(wav2, p=0.1)

            # clean
            # frames2, wav2 = frames, wav
            
            # Fixed (a): run CE loss on the CLEAN pass, not the corrupted pass
            out       = model(frames2, wav2)
            out_clean = model(frames, wav)
            out_corrV = model(corrupt_frames(frames, p=1.0), wav)
            out_corrA = model(frames, corrupt_audio(wav, p=1.0))
 
            loss = ramer_loss(out_clean, y)
 
            r0 = out_clean["r"].detach()
            rV = out_corrV["r"]
            rA = out_corrA["r"]
 
            # loss_corrV = torch.relu(rV[:, 0] - r0[:, 0] + 0.1).mean()
            # loss_corrA = torch.relu(rA[:, 1] - r0[:, 1] + 0.1).mean()
            # # Fixed (b): hinge weight 1.5 (was 0.5 — too weak vs CE ~1.5)
            # loss = loss + 1.5 * (loss_corrV + loss_corrA)

            # In train_ramer, replace hinge with:
            loss_corrV = (torch.relu(rV[:, 0] - r0[:, 0]) ** 2).mean()  # squared, no margin
            loss_corrA = (torch.relu(rA[:, 1] - r0[:, 1]) ** 2).mean()
            loss = loss + 3.0 * (loss_corrV + loss_corrA)
 
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()
 
            total += loss.item()
 
        metrics = eval_ramer_metrics(model, dl_val,
                                     num_classes=model.head_v[-1].out_features,
                                     device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )
 
        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc":   metrics["accuracy"],
                "val_macro": best_f1,
            }, save_path)
            print(f"Saved best RAMER model to {save_path}")
 
    return best_f1

@torch.no_grad()
def eval_ramer_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [9]:
# ============================== EARLY FUSION ===============================

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EarlyFusion(nn.Module):
    """
    Early fusion baseline:
      z_v = VisionBackbone(frames)
      z_a = AudioBackbone(wav)
      z = concat(z_v, z_a)
      logits = classifier(z)
    Returns dict with "logits" to be compatible with your eval code.
    Optionally also returns logits_v/logits_a for auxiliary loss fairness.
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        # Optional unimodal heads (useful for fair auxiliary losses)
        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # Early fusion classifier on concatenated features
        self.fuse = nn.Sequential(
            nn.LayerNorm(z_dim * 2),
            nn.Linear(z_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes)
        )

    def forward(self, frames, wav):
        z_v = self.vision(frames)  # [B,z_dim]
        z_a = self.audio(wav)      # [B,z_dim]

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        z = torch.cat([z_v, z_a], dim=-1)
        logits = self.fuse(z)

        return {"logits": logits, "logits_v": logits_v, "logits_a": logits_a}

In [11]:
def early_fusion_loss(out, y, alpha=0.3, beta=0.3):
    loss_fuse = F.cross_entropy(out["logits"], y)
    loss_uni  = alpha * F.cross_entropy(out["logits_v"], y) + beta * F.cross_entropy(out["logits_a"], y)
    return loss_fuse + loss_uni

from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def train_early_fusion(model, dl_train, dl_val, device="cuda",
                       epochs=5, lr=3e-4,
                       save_path="earlyfusion_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    num_classes = model.fuse[-1].out_features

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # SAME heterogeneity pipeline as you use elsewhere
            frames2 = corrupt_frames(frames, p=0.20)
            wav2    = corrupt_audio(wav, p=0.20)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)
            loss = early_fusion_loss(out, y)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_early_fusion_metrics(model, dl_val, num_classes=num_classes, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best EarlyFusion model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_early_fusion_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)


In [12]:
# ============================ LATE FUSION ====================================

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LateFusion(nn.Module):
    """
    Late fusion baseline = average of unimodal logits.
    logits = 0.5*(logits_v + logits_a)
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

    def forward(self, frames, wav):
        z_v = self.vision(frames)
        z_a = self.audio(wav)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        logits = 0.5 * (logits_v + logits_a)

        return {"logits": logits, "logits_v": logits_v, "logits_a": logits_a}

In [14]:
def latefusion_loss(out, y, alpha=0.3, beta=0.3):
    """
    Fused CE + optional unimodal CE terms (fair + stable).
    """
    loss_fuse = F.cross_entropy(out["logits"], y)
    loss_uni  = alpha * F.cross_entropy(out["logits_v"], y) + beta * F.cross_entropy(out["logits_a"], y)
    return loss_fuse + loss_uni

from torch.utils.data import DataLoader
import torch.optim as optim

def train_latefusion(model, dl_train, dl_val, device="cuda",
                     epochs=12, lr=2e-4,
                     save_path="latefusion_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    num_classes = model.head_v[-1].out_features

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # same heterogeneity augmentation you use
            frames2 = corrupt_frames(frames, p=0.20)
            wav2    = corrupt_audio(wav, p=0.20)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)
            loss = latefusion_loss(out, y)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_latefusion_metrics(model, dl_val, num_classes=num_classes, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best LateFusion model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_latefusion_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y, all_p = [], []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [15]:
# ============================ GATED FUSION ====================================

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedFusion(nn.Module):
    """
    Independent gates (not normalized to sum=1):
      g_v = sigmoid(MLP(z_v)), g_a = sigmoid(MLP(z_a))
      z = g_v*z_v + g_a*z_a
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        # Gates per modality (scalar per sample)
        self.gate_v = nn.Sequential(nn.Linear(z_dim, 1), nn.Sigmoid())
        self.gate_a = nn.Sequential(nn.Linear(z_dim, 1), nn.Sigmoid())

        # Optional unimodal heads (helps stability, fair vs RAMER/ATTN)
        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # Fused classifier
        self.head_fuse = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

    def forward(self, frames, wav):
        z_v = self.vision(frames)   # [B,z_dim]
        z_a = self.audio(wav)       # [B,z_dim]

        g_v = self.gate_v(z_v)      # [B,1]
        g_a = self.gate_a(z_a)      # [B,1]
        g = torch.cat([g_v, g_a], dim=1)       # [B,2]
        w = torch.softmax(g, dim=1)            # [B,2] sum-to-1
        w_v, w_a = w[:,0:1], w[:,1:2]
        z = w_v * z_v + w_a * z_a
        
        logits = self.head_fuse(z)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        return {
            "logits": logits,
            "logits_v": logits_v,
            "logits_a": logits_a,
            "g_v": g_v,
            "g_a": g_a,
            "w": w
        }

In [17]:
def gated_loss(out, y, alpha=0.3, beta=0.3, gate_reg=0.01):
    logits = out["logits"]
    logits_v = out["logits_v"]
    logits_a = out["logits_a"]
    g_v = out["g_v"]
    g_a = out["g_a"]

    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha*F.cross_entropy(logits_v, y) + beta*F.cross_entropy(logits_a, y)

    # Gate regularizer: mild penalty if gates saturate too hard (keeps training stable)
    # penalize distance from 0.5
    reg = ((g_v - 0.5).abs().mean() + (g_a - 0.5).abs().mean())
    loss_reg = gate_reg * reg

    return loss_fuse + loss_uni + loss_reg

from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def train_gated(model, dl_train, dl_val, device="cuda",
                epochs=12, lr=2e-4,
                save_path="gated_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    num_classes = model.head_fuse[-1].out_features

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # Same heterogeneity pipeline (you can lower p_drop for fairness if you want)
            frames2 = corrupt_frames(frames, p=0.25)
            wav2    = corrupt_audio(wav, p=0.25)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.10)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)
            loss = gated_loss(out, y, alpha=0.3, beta=0.3, gate_reg=0.01)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_gated_metrics(model, dl_val, num_classes=num_classes, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best GatedFusion model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_gated_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y, all_p = [], []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [18]:
# ============================ MAIN START =====================================

In [19]:
# Step 1: Build unified label map
label_map = build_label_map([
    os.path.join('datasets', 'affectNet', 'updated_labels.csv'),
    os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'),
    # "/content/prep/ravdess/manifest.jsonl",
])
print(label_map)


{'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4}


In [20]:
# Step 2: DataLoaders

from torch.utils.data import DataLoader

# CREMA-D AV
ds_cre_tr = AVManifestDataset(os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'), "train", label_map, num_frames=8)
ds_cre_va = AVManifestDataset(os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'), "val", label_map, num_frames=8)
dl_cre_tr = DataLoader(ds_cre_tr, batch_size=12, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
dl_cre_va = DataLoader(ds_cre_va, batch_size=12, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)

# RAVDESS AV
# 6 speakers test
# ds_rav_te = AVManifestDataset(os.path.join('datasets', 'ravdess-prep', 'manifest.jsonl'), "test", label_map, num_frames=8)
# all speakers test
ds_rav_te = AVManifestDataset(os.path.join('datasets', 'ravdess-prep', 'manifest_all_test.jsonl'), "test", label_map, num_frames=8)
dl_rav_te = DataLoader(ds_rav_te, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

n_classes = len(label_map)

In [21]:
# # Step 4: Initialize RAMER and (optionally) load visual weights
# ramer = RAMER(n_classes=n_classes, z_dim=256)

# # Step 5: Train RAMER on CREMA-D (A/V) with heterogeneity
# best_val = train_ramer(ramer, dl_cre_tr, dl_cre_va, epochs=15, lr=1e-4)

In [22]:
%%capture
# Load RAMER model
# ramer = RAMER(n_classes=n_classes)
# ckpt = torch.load(os.path.join("models", "ramer_best_3_epochs.pt"), map_location="cpu")
# ramer.load_state_dict(ckpt["model_state"])
# ramer.to(device).eval()

# acc_rav = eval_ramer_metrics(ramer, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS ramer audio cnn acc:", acc_rav)

In [23]:
# ==================================== EARLY FUSION MODEL ===============================

In [24]:
# early = EarlyFusion(n_classes=len(label_map), z_dim=256)
# early.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True
# )

# # train_early_fusion(early, dl_cre_tr, dl_cre_va, epochs=12, lr=2e-4, save_path="early_best.pt")

# # load early fusion model
# early = EarlyFusion(n_classes=n_classes) 
# ckpt_early = torch.load(os.path.join("models", "early_best.pt"), map_location="cpu") 
# early.load_state_dict(ckpt_early["model_state"]) 
# early.to(device).eval()

# acc_early = eval_early_fusion_metrics(early, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS early audio cnn acc:", acc_early)

In [25]:
# ================================= LATE FUSION MODEL ===================================

In [26]:
# late = LateFusion(n_classes=len(label_map), z_dim=256)
# late.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True
# )
# # train_latefusion(late, dl_cre_tr, dl_cre_va, epochs=12, lr=2e-4, save_path="latefusion_best.pt")

# # load 
# late = LateFusion(n_classes=n_classes) 
# ckpt_late = torch.load(os.path.join("models", "latefusion_best.pt"), map_location="cpu") 
# late.load_state_dict(ckpt_late["model_state"]) 
# late.to(device).eval()

# acc_late = eval_latefusion_metrics(late, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS late audio cnn acc:", acc_late)

In [27]:
# ================================= GATE FUSION MODEL ===================================

In [28]:
# gated = GatedFusion(n_classes=len(label_map), z_dim=256)
# gated.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True
# )
# # train_gated(gated, dl_cre_tr, dl_cre_va, epochs=12, lr=2e-4, save_path="gated_best.pt")

# # load
# gated = GatedFusion(n_classes=n_classes) 
# ckpt_gate = torch.load(os.path.join("models", "gated_best.pt"), map_location="cpu") 
# gated.load_state_dict(ckpt_gate["model_state"]) 
# gated.to(device).eval()

# acc_gate = eval_gated_metrics(gated, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS gate audio cnn acc:", acc_gate)

In [29]:
# metrics_early_rav = eval_early_fusion_metrics(early, dl_rav_te, num_classes=len(label_map))
# print("EarlyFusion zero-shot on RAVDESS:", metrics_early_rav)

In [30]:
# ================================= ATTENTION MODEL =======================================

In [31]:
@torch.no_grad()
def attn_weight_stats(model, dl, device="cuda", max_batches=10):
    model.eval()
    ws = []
    for i, b in enumerate(dl):
        if i >= max_batches:
            break
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        out = model(frames, wav)
        ws.append(out["w"].detach().cpu())
    w = torch.cat(ws, 0)
    return w.mean(0).tolist(), w.std(0).tolist()

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionFusion(nn.Module):
    """
    Standard attention fusion baseline:
      z_v, z_a -> w = softmax(MLP([z_v,z_a])) -> z = wv*z_v + wa*z_a -> logits
    Also returns logits_v/logits_a for fair auxiliary supervision (optional).
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()

        # Reuse your backbones if you want, or default to your existing ones
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        # modality-specific heads (optional but recommended for fair comparison)
        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # attention weights over modalities
        self.attn = nn.Sequential(
            nn.Linear(z_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, 2)   # w_v, w_a
        )

        # fused classifier head
        self.head_fuse = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

    def forward(self, frames, wav):
        # frames: [B,T,3,224,224], wav: [B,N]
        z_v = self.vision(frames)
        z_a = self.audio(wav)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        w_logits = self.attn(torch.cat([z_v, z_a], dim=-1))
        w = torch.softmax(w_logits, dim=-1)  # [B,2]
        w_v, w_a = w[:, 0:1], w[:, 1:2]

        z = w_v * z_v + w_a * z_a
        logits = self.head_fuse(z)

        return {
            "logits": logits,
            "logits_v": logits_v,
            "logits_a": logits_a,
            "w": w
        }

In [33]:
class LogitAttentionFusion(nn.Module):
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        self.attn = nn.Sequential(
            nn.Linear(z_dim*2, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, frames, wav):
        z_v = self.vision(frames)
        z_a = self.audio(wav)
        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)
        w = torch.softmax(self.attn(torch.cat([z_v, z_a], dim=-1)), dim=-1)
        logits = w[:,0:1]*logits_v + w[:,1:2]*logits_a
        return {"logits": logits, "logits_v": logits_v, "logits_a": logits_a, "w": w}

In [34]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def attn_loss(out, y, alpha=0.3, beta=0.3, ent_lambda=0.01):
    logits, logits_v, logits_a = out["logits"], out["logits_v"], out["logits_a"]
    w = out["w"]  # [B,2]

    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha * F.cross_entropy(logits_v, y) + beta * F.cross_entropy(logits_a, y)

    # Encourage non-degenerate weights (optional)
    ent = -(w * (w.clamp_min(1e-8).log())).sum(dim=-1).mean()
    # loss_ent = -ent_lambda * ent
    loss_ent = -0.05 * ent   # increase from 0.01 to 0.05

    return loss_fuse + loss_uni + loss_ent

def train_attention(model, dl_train, dl_val, device="cuda",
                    epochs=5, lr=3e-4,
                    save_path="attn_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0

    # --- NEW: infer num_classes from a single forward pass ---
    model.eval()
    b0 = next(iter(dl_val))
    with torch.no_grad():
        out0 = model(b0["frames"].to(device), b0["audio"].to(device))
        num_classes = out0["logits"].shape[-1]
    # --------------------------------------------------------

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # heavy heterogeneity
            # frames2 = corrupt_frames(frames, p=0.20)
            # wav2    = corrupt_audio(wav, p=0.20)
            # frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            # wav2    = temporal_misalign(wav2, p=0.1)

            # 0 corruption/clean values for benchmark
            frames2 = frames
            wav2    = wav

            out = model(frames2, wav2)
            loss = attn_loss(out, y, alpha=0.3, beta=0.3, ent_lambda=0.05)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_attention_metrics(model, dl_val, num_classes=num_classes, device=device)

        # NEW: optional diagnostics only if model returns "w"
        if "w" in out0:
            mean_w, std_w = attn_weight_stats(model, dl_val, device=device)
            print(
                f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
                f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f} | "
                f"attn_mean={mean_w} attn_std={std_w}"
            )
        else:
            print(
                f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
                f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
            )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best Attention model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_attention_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [35]:
# 2) Train AttentionFusion baseline (same backbones/settings)
# attn = AttentionFusion(n_classes=len(label_map), z_dim=256)
# attn.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True)
# train_attention(attn, dl_cre_tr, dl_cre_va, epochs=20, lr=1e-4, save_path="attn_best.pt")


# Train attention logits
# attn_logit = LogitAttentionFusion(n_classes=len(label_map), z_dim=256)
# use default weights, uncomment below line to use affect-net weights
# attn_logit.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True)
# train_attention(attn_logit, dl_cre_tr, dl_cre_va, epochs=20, lr=1e-4, save_path="attn_logit_best.pt")



# 3) Zero-shot eval on RAVDESS test
# m_ramer = eval_ramer_metrics(ramer, dl_rav_te, num_classes=len(label_map))
# m_attn  = eval_attention_metrics(attn, dl_rav_te, num_classes=len(label_map))

# print("RAMER  zero-shot:", m_ramer)
# print("\nATTN   zero-shot:", m_attn)

In [36]:
# Load attention models
# attn = AttentionFusion(n_classes=n_classes) 
# ckpt = torch.load(os.path.join("models", "attn_best.pt"), map_location="cpu") 
# attn.load_state_dict(ckpt["model_state"]) 
# attn.to(device).eval()

# attn_logit = LogitAttentionFusion(n_classes=n_classes) 
# ckpt_logit = torch.load(os.path.join("models", "attn_logit_best.pt"), map_location="cpu") 
# attn_logit.load_state_dict(ckpt_logit["model_state"]) 
# attn_logit.to(device).eval()

# m_attn = eval_attention_metrics(attn, dl_rav_te, num_classes=len(label_map))
# print("\nATTN zero-shot:", m_attn)

# m_attn_logit = eval_attention_metrics(attn_logit, dl_rav_te, num_classes=len(label_map))
# print("\nATTN Logits zero-shot:", m_attn_logit)

In [37]:
# #  ============================= EXPERIMENTS ZERO-SHOT TEST ==================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TESTING FOT SEED 42, 150, 2026 

attn_logit_aug_default_weights = LogitAttentionFusion(n_classes=n_classes)
ckpt1 = torch.load(os.path.join("multi_seed_models", "attn_logit_aug_best_42.pt"), map_location="cpu") 
attn_logit_aug_default_weights.load_state_dict(ckpt1["model_state"]) 
attn_logit_aug_default_weights.to(device).eval()

attn_logit_clean_default_weights = LogitAttentionFusion(n_classes=n_classes)
ckpt2 = torch.load(os.path.join("multi_seed_models", "attn_logit_clean_best_42.pt"), map_location="cpu") 
attn_logit_clean_default_weights.load_state_dict(ckpt2["model_state"]) 
attn_logit_clean_default_weights.to(device).eval()

# # ------------------------------------------------------------------------------------------------------------------------

ramer_aug_default_weights = RAMER(n_classes=n_classes)
ckpt3 = torch.load(os.path.join("multi_seed_models", "ramer_aug_best_2026.pt"), map_location="cpu") 
ramer_aug_default_weights.load_state_dict(ckpt3["model_state"]) 
ramer_aug_default_weights.to(device).eval()

ramer_clean_default_weights = RAMER(n_classes=n_classes)
ckpt4 = torch.load(os.path.join("multi_seed_models", "ramer_clean_best_2026.pt"), map_location="cpu") 
ramer_clean_default_weights.load_state_dict(ckpt4["model_state"]) 
ramer_clean_default_weights.to(device).eval()

# # ---------------------------------------------------------------------------------------------------------------------

gated_aug_default_weights = GatedFusion(n_classes=n_classes)
ckpt5 = torch.load(os.path.join("multi_seed_models", "gated_aug_best_42.pt"), map_location="cpu") 
gated_aug_default_weights.load_state_dict(ckpt5["model_state"]) 
gated_aug_default_weights.to(device).eval()

gated_clean_default_weights = GatedFusion(n_classes=n_classes)
ckpt6 = torch.load(os.path.join("multi_seed_models", "gated_clean_best_42.pt"), map_location="cpu") 
gated_clean_default_weights.load_state_dict(ckpt6["model_state"]) 
gated_clean_default_weights.to(device).eval()

# # -------------------------------------------------------------------------------------------------------------

late_aug_default_weights = LateFusion(n_classes=n_classes)
ckpt7 = torch.load(os.path.join("multi_seed_models", "late_aug_best_42.pt"), map_location="cpu") 
late_aug_default_weights.load_state_dict(ckpt7["model_state"]) 
late_aug_default_weights.to(device).eval()

late_clean_default_weights = LateFusion(n_classes=n_classes)
ckpt8 = torch.load(os.path.join("multi_seed_models", "late_clean_best_42.pt"), map_location="cpu") 
late_clean_default_weights.load_state_dict(ckpt8["model_state"]) 
late_clean_default_weights.to(device).eval()

# # -------------------------------------------------------------------------------------------------------------

early_aug_default_weights = EarlyFusion(n_classes=n_classes)
ckpt9 = torch.load(os.path.join("multi_seed_models", "early_aug_best_42.pt"), map_location="cpu") 
early_aug_default_weights.load_state_dict(ckpt9["model_state"]) 
early_aug_default_weights.to(device).eval()

early_clean_default_weights = EarlyFusion(n_classes=n_classes)
ckpt10 = torch.load(os.path.join("multi_seed_models", "early_clean_best_42.pt"), map_location="cpu") 
early_clean_default_weights.load_state_dict(ckpt10["model_state"]) 
early_clean_default_weights.to(device).eval()

'''
r1 = eval_attention_metrics(attn_logit_aug_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nAttention logit aug with default weights zero-shot:", r1)

r2 = eval_attention_metrics(attn_logit_clean_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nAttention logit clean with default weights zero-shot:", r2)

'''
r3 = eval_ramer_metrics(ramer_aug_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nRAMER aug with default weights zero-shot:", r3)

r4 = eval_ramer_metrics(ramer_clean_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nRAMER clean with default weights zero-shot:", r4)

'''
r5 = eval_early_fusion_metrics(early_aug_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nEarly aug with default weights zero-shot:", r5)

r6 = eval_early_fusion_metrics(early_clean_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nEarly clean with default weights zero-shot:", r6)

r7 = eval_latefusion_metrics(late_aug_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nLate aug with default weights zero-shot:", r7)

r8 = eval_latefusion_metrics(late_clean_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nLate clean with default weights zero-shot:", r8)

r9 = eval_gated_metrics(gated_aug_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nGate aug with default weights zero-shot:", r9)

r10 = eval_gated_metrics(gated_clean_default_weights, dl_rav_te, num_classes=len(label_map))
print("\nGate clean with default weights zero-shot:", r10)

'''

C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



RAMER aug with default weights zero-shot: {'accuracy': 0.36770832538604736, 'macro_f1': 0.2751595973968506, 'macro_precision': 0.6168642044067383, 'macro_recall': 0.36770835518836975, 'per_class_precision': [0.4382978677749634, 0.5913978219032288, 0.75, 0.3046252131462097, 1.0], 'per_class_recall': [0.5364583134651184, 0.2864583432674408, 0.015625, 0.9947916865348816, 0.0052083334885537624], 'per_class_f1': [0.48243558406829834, 0.38596493005752563, 0.030612245202064514, 0.4664224684238434, 0.010362694039940834], 'confusion_matrix': tensor([[103,   8,   1,  80,   0],
        [ 17,  55,   0, 120,   0],
        [ 68,  11,   3, 110,   0],
        [  1,   0,   0, 191,   0],
        [ 46,  19,   0, 126,   1]])}

RAMER clean with default weights zero-shot: {'accuracy': 0.390625, 'macro_f1': 0.2847714424133301, 'macro_precision': 0.5296967625617981, 'macro_recall': 0.390625, 'per_class_precision': [0.3971830904483795, 0.5970149040222168, 0.5, 0.35428571701049805, 0.800000011920929], 'per_cla

'\nr5 = eval_early_fusion_metrics(early_aug_default_weights, dl_rav_te, num_classes=len(label_map))\nprint("\nEarly aug with default weights zero-shot:", r5)\n\nr6 = eval_early_fusion_metrics(early_clean_default_weights, dl_rav_te, num_classes=len(label_map))\nprint("\nEarly clean with default weights zero-shot:", r6)\n\nr7 = eval_latefusion_metrics(late_aug_default_weights, dl_rav_te, num_classes=len(label_map))\nprint("\nLate aug with default weights zero-shot:", r7)\n\nr8 = eval_latefusion_metrics(late_clean_default_weights, dl_rav_te, num_classes=len(label_map))\nprint("\nLate clean with default weights zero-shot:", r8)\n\nr9 = eval_gated_metrics(gated_aug_default_weights, dl_rav_te, num_classes=len(label_map))\nprint("\nGate aug with default weights zero-shot:", r9)\n\nr10 = eval_gated_metrics(gated_clean_default_weights, dl_rav_te, num_classes=len(label_map))\nprint("\nGate clean with default weights zero-shot:", r10)\n\n'

In [38]:
# ================================= EXTREME STRESS TEST ==========================================

In [39]:
import random
import torch
import torch.nn.functional as F

def audio_silence(wav):
    return torch.zeros_like(wav)

def audio_clip(wav, clip_val=0.2):
    return torch.clamp(wav, -clip_val, clip_val)

def audio_packet_loss(wav, drop_prob=0.3, chunk=400):  # chunk ~25ms at 16kHz
    # Randomly zero small chunks
    wav2 = wav.clone()
    B, N = wav2.shape
    num_chunks = max(1, N // chunk)
    mask = (torch.rand(B, num_chunks, device=wav.device) > drop_prob).float()
    mask = mask.repeat_interleave(chunk, dim=1)[:, :N]
    return wav2 * mask

def audio_add_noise_snr(wav, snr_db=0.0):
    """
    Add gaussian noise to reach target SNR in dB.
    snr_db = 0 is very harsh, -5 even harsher.
    """
    # wav: [B,N]
    eps = 1e-8
    sig_power = wav.pow(2).mean(dim=1, keepdim=True) + eps
    noise = torch.randn_like(wav)
    noise_power = noise.pow(2).mean(dim=1, keepdim=True) + eps
    snr_lin = 10 ** (snr_db / 10.0)
    scale = torch.sqrt(sig_power / (snr_lin * noise_power))
    return torch.clamp(wav + noise * scale, -1.0, 1.0)

def video_black(frames):
    return torch.zeros_like(frames)

def video_heavy_occlusion(frames, occ_ratio=0.7):
    """
    Zero out a big rectangle region on all frames.
    frames: [B,T,3,H,W]
    """
    B,T,C,H,W = frames.shape
    frames2 = frames.clone()
    occ_area = int(H * W * occ_ratio)
    # choose rectangle dims roughly
    rect_h = random.randint(int(H*0.5), H)
    rect_w = max(1, occ_area // max(rect_h,1))
    rect_w = min(rect_w, W)
    y0 = random.randint(0, max(0, H-rect_h))
    x0 = random.randint(0, max(0, W-rect_w))
    frames2[:, :, :, y0:y0+rect_h, x0:x0+rect_w] = 0.0
    return frames2

def video_downsample(frames, scale=0.25):
    """
    Severe resolution loss: downsample then upsample.
    """
    B,T,C,H,W = frames.shape
    h2 = max(1, int(H * scale))
    w2 = max(1, int(W * scale))
    x = frames.view(B*T, C, H, W)
    x = F.interpolate(x, size=(h2,w2), mode="bilinear", align_corners=False)
    x = F.interpolate(x, size=(H,W), mode="bilinear", align_corners=False)
    return x.view(B, T, C, H, W)

def video_motion_blur(frames, k=9):
    """
    Approx motion blur: average pool along width.
    """
    B,T,C,H,W = frames.shape
    x = frames.view(B*T, C, H, W)
    # blur kernel along width
    x = F.avg_pool2d(x, kernel_size=(1,k), stride=1, padding=(0,k//2))
    return x.view(B, T, C, H, W)


In [40]:
@torch.no_grad()
def eval_model_metrics_logits(model, dl, num_classes, device="cuda"):
    model.eval().to(device)
    all_y, all_p = [], []

    for b in dl:
        frames = b["frames"].to(device, non_blocking=True)
        wav    = b["audio"].to(device, non_blocking=True)
        y      = b["label"].to(device, non_blocking=True)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)


@torch.no_grad()
def extreme_stress_test(model, dl, num_classes, device="cuda"):
    """
    Extreme conditions designed to break standard fusion.
    Returns dict: condition -> metrics dict
    """
    model.eval().to(device)

    results = {}
    conditions = [
        "clean",
        "audio_silence",
        "audio_snr0",
        "audio_snr_minus5",
        "audio_packetloss_30",
        "audio_clip_0p2",
        "video_black",
        "video_occlusion_30",
        "video_occlusion_70",
        "video_downsample_25",
        "video_motionblur_9",
        "combo_audio_snr0_video_occ70",
        "combo_audio_packetloss_video_downsample",
    ]

    for cond in conditions:
        all_y, all_p = [], []
        for b in dl:
            frames = b["frames"].to(device, non_blocking=True)
            wav    = b["audio"].to(device, non_blocking=True)
            y      = b["label"].to(device, non_blocking=True)

            # Apply corruption
            if cond == "clean":
                pass
            elif cond == "audio_silence":
                wav = audio_silence(wav)
            elif cond == "audio_snr0":
                wav = audio_add_noise_snr(wav, snr_db=0.0)
            elif cond == "audio_snr_minus5":
                wav = audio_add_noise_snr(wav, snr_db=-5.0)
            elif cond == "audio_packetloss_30":
                wav = audio_packet_loss(wav, drop_prob=0.3, chunk=400)
            elif cond == "audio_clip_0p2":
                wav = audio_clip(wav, clip_val=0.2)

            elif cond == "video_black":
                frames = video_black(frames)
            elif cond == "video_occlusion_30":
                frames = video_heavy_occlusion(frames, occ_ratio=0.3)
            elif cond == "video_occlusion_70":
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)
            elif cond == "video_downsample_25":
                frames = video_downsample(frames, scale=0.25)
            elif cond == "video_motionblur_9":
                frames = video_motion_blur(frames, k=9)

            elif cond == "combo_audio_snr0_video_occ70":
                wav = audio_add_noise_snr(wav, snr_db=0.0)
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)
            elif cond == "combo_audio_packetloss_video_downsample":
                wav = audio_packet_loss(wav, drop_prob=0.3, chunk=400)
                frames = video_downsample(frames, scale=0.25)

            out = model(frames, wav)
            pred = out["logits"].argmax(dim=-1)

            all_y.append(y.detach())
            all_p.append(pred.detach())

        y_true = torch.cat(all_y, dim=0)
        y_pred = torch.cat(all_p, dim=0)
        results[cond] = compute_classification_metrics(y_true, y_pred, num_classes)

        print(f"[{cond}] acc={results[cond]['accuracy']:.4f} macroF1={results[cond]['macro_f1']:.4f}")

    return results

In [41]:
# # ----------- EXTREME STRESS TEST WITH DEFAULT WEIGHTS -------------------------------

num_classes = len(label_map)

print("=== RAMER Extreme Stress Test with clean training ===")
ramer_ext_clean = extreme_stress_test(ramer_clean_default_weights, dl_rav_te, num_classes=num_classes)

print("=== RAMER Extreme Stress Test with aug training ===")
ramer_ext_aug = extreme_stress_test(ramer_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== ATTN LOGITS Extreme Stress Test with clean training ===")
# attn_logits_ext_clean = extreme_stress_test(attn_logit_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== ATTN LOGITS Extreme Stress Test with aug training ===")
# attn_logits_ext_aug = extreme_stress_test(attn_logit_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Gated Fusion Extreme Stress Test with clean training ===")
# gated_ext_clean = extreme_stress_test(gated_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Gated Fusion Extreme Stress Test with aug training ===")
# gated_ext_aug = extreme_stress_test(gated_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Late Fusion Extreme Stress Test with clean training ===")
# late_ext_clean = extreme_stress_test(late_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Late Fusion Extreme Stress Test with aug training ===")
# late_ext_aug = extreme_stress_test(late_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Early Fusion Extreme Stress Test with clean training ===")
# early_ext_clean = extreme_stress_test(early_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Early Fusion Extreme Stress Test with aug training ===")
# early_ext_aug = extreme_stress_test(early_aug_default_weights, dl_rav_te, num_classes=num_classes)


=== RAMER Extreme Stress Test with clean training ===
[clean] acc=0.3969 macroF1=0.2877
[audio_silence] acc=0.3927 macroF1=0.3032
[audio_snr0] acc=0.4313 macroF1=0.3628
[audio_snr_minus5] acc=0.4229 macroF1=0.3531
[audio_packetloss_30] acc=0.3823 macroF1=0.2768
[audio_clip_0p2] acc=0.3896 macroF1=0.2814
[video_black] acc=0.3115 macroF1=0.1964
[video_occlusion_30] acc=0.2833 macroF1=0.2244
[video_occlusion_70] acc=0.2198 macroF1=0.1485
[video_downsample_25] acc=0.2510 macroF1=0.1657
[video_motionblur_9] acc=0.3344 macroF1=0.2452
[combo_audio_snr0_video_occ70] acc=0.2719 macroF1=0.2292
[combo_audio_packetloss_video_downsample] acc=0.2438 macroF1=0.1537
=== RAMER Extreme Stress Test with aug training ===
[clean] acc=0.3625 macroF1=0.2727
[audio_silence] acc=0.3573 macroF1=0.2793
[audio_snr0] acc=0.3969 macroF1=0.3317
[audio_snr_minus5] acc=0.3990 macroF1=0.3280
[audio_packetloss_30] acc=0.3646 macroF1=0.2696
[audio_clip_0p2] acc=0.3708 macroF1=0.2802
[video_black] acc=0.3135 macroF1=0.189

In [42]:
@torch.no_grad()
def reliability_weight_analysis(model, dl, device="cuda"):
    """
    Analyze how RAMER reliability weights behave under corruption.
    
    Returns:
        dict:
        {
            condition: {
                "mean_rv": float,
                "mean_ra": float,
                "std_rv": float,
                "std_ra": float
            }
        }
    }
    """

    model.eval().to(device)

    conditions = [
        "clean",
        "audio_silence",
        "audio_snr0",
        "audio_snr_minus5",
        "audio_packetloss_30",
        "audio_clip_0p2",
        "video_black",
        "video_occlusion_30",
        "video_occlusion_70",
        "video_downsample_25",
        "video_motionblur_9",
        "combo_audio_snr0_video_occ70",
        "combo_audio_packetloss_video_downsample",
    ]

    results = {}

    for cond in conditions:

        all_rv = []
        all_ra = []

        for b in dl:

            frames = b["frames"].to(device, non_blocking=True)
            wav    = b["audio"].to(device, non_blocking=True)

            # =========================================================
            # APPLY SAME CORRUPTION LOGIC AS YOUR STRESS TEST
            # =========================================================

            if cond == "clean":
                pass

            elif cond == "audio_silence":
                wav = audio_silence(wav)

            elif cond == "audio_snr0":
                wav = audio_add_noise_snr(wav, snr_db=0.0)

            elif cond == "audio_snr_minus5":
                wav = audio_add_noise_snr(wav, snr_db=-5.0)

            elif cond == "audio_packetloss_30":
                wav = audio_packet_loss(wav, drop_prob=0.3, chunk=400)

            elif cond == "audio_clip_0p2":
                wav = audio_clip(wav, clip_val=0.2)

            elif cond == "video_black":
                frames = video_black(frames)

            elif cond == "video_occlusion_30":
                frames = video_heavy_occlusion(frames, occ_ratio=0.3)

            elif cond == "video_occlusion_70":
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)

            elif cond == "video_downsample_25":
                frames = video_downsample(frames, scale=0.25)

            elif cond == "video_motionblur_9":
                frames = video_motion_blur(frames, k=9)

            elif cond == "combo_audio_snr0_video_occ70":
                wav = audio_add_noise_snr(wav, snr_db=0.0)
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)

            elif cond == "combo_audio_packetloss_video_downsample":
                wav = audio_packet_loss(wav, drop_prob=0.3, chunk=400)
                frames = video_downsample(frames, scale=0.25)

            # =========================================================
            # FORWARD
            # =========================================================

            out = model(frames, wav)

            # IMPORTANT:
            # only RAMER has "r"
            if "r" not in out:
                raise ValueError("Model does not output reliability weights 'r'.")

            r = out["r"]   # [B,2]

            rv = r[:,0]
            ra = r[:,1]

            all_rv.append(rv.detach().cpu())
            all_ra.append(ra.detach().cpu())

        all_rv = torch.cat(all_rv)
        all_ra = torch.cat(all_ra)

        results[cond] = {
            "mean_rv": float(all_rv.mean()),
            "mean_ra": float(all_ra.mean()),
            "std_rv": float(all_rv.std()),
            "std_ra": float(all_ra.std()),
        }

        print(
            f"[{cond}] "
            f"rv={results[cond]['mean_rv']:.4f}±{results[cond]['std_rv']:.4f} | "
            f"ra={results[cond]['mean_ra']:.4f}±{results[cond]['std_ra']:.4f}"
        )

    return results

In [43]:
# RAMER - Reliability Vectors Test

print("=== RAMER reliability test with clean training ===")
ramer_rel_clean = reliability_weight_analysis(ramer_clean_default_weights, dl_rav_te)

print("=== RAMER reliability test with aug training ===")
ramer_rel_aug = reliability_weight_analysis(ramer_aug_default_weights, dl_rav_te)
                                         

=== RAMER reliability test with clean training ===
[clean] rv=0.7086±0.0475 | ra=0.2914±0.0475
[audio_silence] rv=0.7295±0.0604 | ra=0.2705±0.0604
[audio_snr0] rv=0.7148±0.0651 | ra=0.2852±0.0651
[audio_snr_minus5] rv=0.7192±0.0628 | ra=0.2808±0.0628
[audio_packetloss_30] rv=0.7147±0.0450 | ra=0.2853±0.0450
[audio_clip_0p2] rv=0.7081±0.0474 | ra=0.2919±0.0474
[video_black] rv=0.5651±0.0278 | ra=0.4349±0.0278
[video_occlusion_30] rv=0.6768±0.0360 | ra=0.3232±0.0360
[video_occlusion_70] rv=0.6274±0.0430 | ra=0.3726±0.0430
[video_downsample_25] rv=0.6963±0.0384 | ra=0.3037±0.0384
[video_motionblur_9] rv=0.6901±0.0363 | ra=0.3099±0.0363
[combo_audio_snr0_video_occ70] rv=0.6190±0.0389 | ra=0.3810±0.0389
[combo_audio_packetloss_video_downsample] rv=0.6996±0.0347 | ra=0.3004±0.0347
=== RAMER reliability test with aug training ===
[clean] rv=0.7139±0.0451 | ra=0.2861±0.0451
[audio_silence] rv=0.7262±0.0598 | ra=0.2738±0.0598
[audio_snr0] rv=0.7132±0.0659 | ra=0.2868±0.0659
[audio_snr_minus5] r